# 🧱 Template Notebook — Migración Alteryx → PySpark (Fabric)
**Versión:** MVP v0.2.0 · **Fecha:** 2025-10-30  
**Cambios clave:** Auto-cast para ordenaciones, limpieza semántica, QA de conteos (input/output)

> Este notebook es generado automáticamente por el GPT. Estructura estándar: **SETUP, INPUTS, READ, TRANSFORMS, QA, OUTPUTS, LOGGING**.


## 1) SETUP

In [ ]:
# Importaciones base
from pyspark.sql import functions as F
from pyspark.sql.window import Window as W
from pyspark.sql import DataFrame
from datetime import datetime
import json
print('[INFO] Notebook Template v0.2.0 cargado')


## 2) INPUTS (parameters)

In [ ]:
# Parámetros — pueden venir de Pipeline/Notebook params o definirse manualmente
params = {
    'modo': 'lenient',                      # 'strict' | 'lenient'
    'lakehouse_name': 'LakehouseDemo',
    'inputs': {
        # Ejemplos (rellenar por el generador):
        # 'ventas': {'type': 'delta', 'path': 'Files/bronze/ventas'},
        # 'clientes': {'type': 'table', 'name': 'lakehouse.default.clientes'},
    },
    'output': {
        'type': 'delta',                   # 'delta' | 'table'
        'path': 'Files/silver/output_etl', # si 'table', usar 'name'
        # 'name': 'lakehouse.default.output_etl'
    },
    'qa': {
        'enabled': True,
        'threshold_diff': 0.001,          # 0.1% para QA_INPUT_OUTPUT_COUNT
        'required_cols': ['id'],          # ejemplo
        'date_cols': ['fecha_dt']
    },
    'order_columns': []                   # El generador puede inyectar columnas de ordenación
}

# Intentar cargar parámetros externos si existen (Fabric/Databricks pattern safe)
try:
    from notebookutils import mssparkutils  # Fabric
    p = mssparkutils.notebook.getContext().notebookParams
    if p: params.update(p)
    print('[INFO] Parámetros leídos desde notebook context')
except Exception:
    pass

print('[INFO] Modo:', params.get('modo'))


## 3) READ — utilidades y lectura de fuentes

In [ ]:
# Utilidades de lectura/escritura (subset de 04_snippets v0.2.0)
def read_table(table_fqn: str) -> DataFrame:
    return spark.read.table(table_fqn)

def read_delta_path(path: str) -> DataFrame:
    return spark.read.format('delta').load(path)

def read_csv_path(path: str, header: bool = True, inferSchema: bool = True) -> DataFrame:
    return spark.read.option('header', header).option('inferSchema', inferSchema).csv(path)

def write_delta(df: DataFrame, path: str, mode: str = 'overwrite', partitionBy: list[str] | None = None):
    w = df.write.format('delta').mode(mode)
    if partitionBy:
        w = w.partitionBy(*partitionBy)
    w.save(path)

def save_as_table(df: DataFrame, table_fqn: str, mode: str = 'overwrite', partitionBy: list[str] | None = None):
    w = df.write.mode(mode)
    if partitionBy:
        w = w.partitionBy(*partitionBy)
    w.saveAsTable(table_fqn)

# Lectura de fuentes definidas en params['inputs'] → crea un dict de DataFrames
sources = {}
for name, cfg in params.get('inputs', {}).items():
    t = (cfg or {}).get('type')
    try:
        if t == 'table':
            sources[name] = read_table(cfg['name'])
        elif t == 'delta':
            sources[name] = read_delta_path(cfg['path'])
        elif t == 'csv':
            sources[name] = read_csv_path(cfg['path'])
        else:
            print(f"[WARN] Tipo de input no soportado: {t} → {name}")
    except Exception as e:
        print(f"[ERR] No se pudo leer {name}: {str(e)}")

# Heurística: elegir primer DataFrame como df_in si solo hay 1
df_in = None
if len(sources) == 1:
    df_in = list(sources.values())[0]
    print('[INFO] df_in asignado automáticamente desde única fuente')
else:
    print('[INFO] Múltiples inputs detectados. El generador debe ensamblar df_in explícitamente.')


## 4) TRANSFORMS

In [ ]:
# Funciones v0.2.0 (fallbacks seguros si no se importaron desde snippets)
try:
    auto_cast_for_sort
except NameError:
    import re
    def auto_cast_for_sort(df: DataFrame, cols: list[str]) -> DataFrame:
        for c in cols or []:
            try:
                sample = df.select(c).limit(100).toPandas()[c].astype(str)
                if all(re.match(r'^[0-9]+$', x) for x in sample if x not in ('None','nan')):
                    df = df.withColumn(c, F.col(c).cast('int'))
                    print(f"[INFO_CAST_APPLIED] Columna {c} convertida a int para ordenación.")
            except Exception as e:
                print(f"[WARN_CAST_SKIP] No se pudo evaluar columna {c}: {str(e)}")
        return df

try:
    cleanup_redundant_vars
except NameError:
    def cleanup_redundant_vars(locals_dict: dict):
        to_delete = [k for k, v in locals_dict.items() if isinstance(v, DataFrame) and k.startswith('df_temp_')]
        for k in to_delete:
            del locals_dict[k]
            print(f"[CLEANUP] Variable temporal eliminada: {k}")

# ------------------
# 🔧 A partir de aquí, el generador inserta las transformaciones PySpark del flujo
# Ejemplo mínimo: si solo hay un input, la salida es igual a la entrada
df_out = df_in if 'df_in' in globals() and df_in is not None else None

# Aplicar auto-cast si hay columnas de orden declaradas
order_cols = params.get('order_columns', [])
if df_out is not None and order_cols:
    df_out = auto_cast_for_sort(df_out, order_cols)

# Limpieza semántica (solo en modo lenient)
if params.get('modo','lenient').lower() == 'lenient':
    try:
        cleanup_redundant_vars(locals())
    except Exception as e:
        print('[INFO] Limpieza semántica no aplicada:', str(e))


## 5) QA — mínimo (v0.2.0)

In [ ]:
# QA_ROW_COUNT
row_count = None
if 'df_out' in globals() and df_out is not None:
    row_count = df_out.count()
    print('[QA] Row count:', row_count)
else:
    print('[QA][WARN] df_out no definido')

# QA_INPUT_OUTPUT_COUNT
try:
    if params.get('qa',{}).get('enabled', True):
        thr = float(params.get('qa',{}).get('threshold_diff', 0.001))
        cnt_in = (df_in.count() if 'df_in' in globals() and df_in is not None else None)
        cnt_out = (df_out.count() if 'df_out' in globals() and df_out is not None else None)
        if cnt_in is not None and cnt_out is not None:
            diff = abs(cnt_in - cnt_out) / max(cnt_in, 1)
            if diff > thr:
                print(f"[QA][WARN] QA_INPUT_OUTPUT_COUNT_DIFF {diff*100:.2f}% (in={cnt_in} vs out={cnt_out})")
except Exception as e:
    print('[QA][INFO] QA_INPUT_OUTPUT_COUNT no ejecutado:', str(e))

# QA_NOT_NULL_KEYS
required_cols = params.get('qa',{}).get('required_cols', [])
for c in required_cols:
    try:
        nulls = df_out.filter(F.col(c).isNull()).count()
        if nulls > 0:
            print(f"[QA][WARN] NULLs en {c}: {nulls}")
    except Exception as e:
        print(f"[QA][INFO] No se pudo validar not-null para {c}: {str(e)}")

# QA_DATE_FORMAT (si se usa to_date/to_timestamp)
for c in params.get('qa',{}).get('date_cols', []):
    try:
        invalid = df_out.filter(F.col(c).isNull()).count()
        if invalid > 0:
            print(f"[QA][WARN] Formato de fecha inválido en {c}: {invalid}")
    except Exception:
        pass


## 6) OUTPUTS

In [ ]:
# Escritura parametrizada
out_cfg = params.get('output', {})
if 'df_out' in globals() and df_out is not None:
    try:
        if out_cfg.get('type') == 'table' and out_cfg.get('name'):
            save_as_table(df_out, out_cfg['name'], mode='overwrite')
            print('[OUT] Guardado como tabla:', out_cfg['name'])
        else:
            write_delta(df_out, out_cfg.get('path', 'Files/silver/output_etl'), mode='overwrite')
            print('[OUT] Guardado en delta path:', out_cfg.get('path'))
    except Exception as e:
        print('[ERR] No se pudo guardar salida:', str(e))
else:
    print('[OUT][WARN] No hay df_out para guardar')


## 7) LOGGING / EXIT PAYLOAD

In [ ]:
payload = {
    'status': 'SUCCESS' if row_count is not None else 'REVIEW',
    'warnings': 0,                         # el generador puede contar [WARN]
    'todos': 0,                            # TODOs inyectados por modo lenient
    'rows_processed': int(row_count or 0),
    'outputs': {
        'lakehouse_table': out_cfg.get('name',''),
        'path': out_cfg.get('path','Files/silver/output_etl')
    },
    'metrics': {
        'parity_index': None,              # calculado si hay dataset de referencia
        'row_count_match': None            # setear en pipeline si corresponde
    },
    'timestamp': datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
}
print('[EXIT]', json.dumps(payload, ensure_ascii=False))

# Salida para Fabric (opcional)
try:
    from notebookutils import mssparkutils
    mssparkutils.notebook.exit(json.dumps(payload))
except Exception:
    pass
